<a href="https://colab.research.google.com/github/priyal6/General-llm/blob/main/prompt_mlflow.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
!pip install -q mlflow openai

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.7/49.7 kB 3.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.5/50.5 kB 4.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.6/12.6 MB 105.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.5/3.5 MB 114.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 77.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 147.8/147.8 kB 14.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 10.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 212.0/212.0 kB 16.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 94.9/94.9 kB 8.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132.2/132.2 kB 10.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 939.7/939.7 kB 54.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [1]:
import os
from google.colab import userdata

os.environ['OPENAI_API_KEY'] = userdata.get('OPENAI_API_KEY')

print('Setup complete! You can now run MLflow Prompt Registry code')

Setup complete! You can now run MLflow Prompt Registry code


In [3]:
import mlflow

initial_template = """
You are a helpful customer support assistance for {{company_name}}
Please answer the following user question in a friendly tone.

Question: {{question}}
"""

prompt_v1 = mlflow.genai.register_prompt(
    name="customer_support_assistant",
    template=initial_template,
    commit_message="Initial prompt setup with friendly tone guidelines.",
    tags={
        "team": "customer-success",
        "language": "en"
    }
)

print(f"Registered: {prompt_v1.name} (Version {prompt_v1.version})")

2026/06/21 09:30:59 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/06/21 09:30:59 INFO mlflow.store.db.utils: Updating database tables


Registered: customer_support_assistant (Version 1)


In [4]:
updated_template= """
You are a helpful customer support assistant for {{company_name}}.
Answer the user question using bullet points and keep it under 3 sentences.

Question: {{question}}
"""

prompt_v2 = mlflow.genai.register_prompt(
    name="customer_support_assistant",
    template=updated_template,
    commit_message="Updated to enforce concise, bulleted responses.",
    tags={
        "team": "customer-success",
        "language": "en"
    }
)

print(f"Registered: {prompt_v2.name} (Version {prompt_v2.version})")

Registered: customer_support_assistant (Version 2)


In [5]:
mlflow.genai.set_prompt_alias(
    name="customer_support_assistant",
    version=2,
    alias="production"
)

print("Alias 'production' successfully mapped to Version 2.")

Alias 'production' successfully mapped to Version 2.


In [6]:
import os
from openai import OpenAI

client = OpenAI(api_key=os.environ.get("OPENAI_API_KEY"))

production_prompt = mlflow.genai.load_prompt("prompts:/customer_support_assistant@production")

formatted_prompt = production_prompt.format(
    company_name="TechCorp",
    question="How do I reset my account password?"
)

response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {
            "role": "user",
            "content": formatted_prompt
        }
    ],
    temperature=0.2
)

print(response.choices[0].message.content)

- Go to the TechCorp login page and click on "Forgot Password?"  
- Enter your registered email address and follow the instructions in the email you receive.  
- Create a new password and log in with your updated credentials.


In [9]:
import mlflow

rag_chat_template = [
    {
        "role": "system",
        "content": "You are a professional research assistant. Answer the user's question accurately using ONLY the provided reference documents."
    },
    {
          "role": "user",
          "content": """
  Context documents:
  {% for doc in documents %}
  - [ID: {{ doc.id }}] {{ doc.text }}
  {% endfor %}

  Question: {{ user_query }}
  """

    }
]

rag_prompt_v1 = mlflow.genai.register_prompt(
    name="rag_research_assistant",
    template=rag_chat_template,
    commit_message="Initial RAG prompt supporting dynamic document context loops.",
    tags={"pipeline": "rag", "engine": "jinja2"}
)

print(f"Registered: {rag_prompt_v1.name} (Version {rag_prompt_v1.version})")



Registered: rag_research_assistant (Version 1)


In [10]:
updated_rag_chat_template = [
    {
        "role": "system",
        "content": "You are a strict research assistant. Answer the user's question using ONLY the provided documents. If the answer cannot be found in the documents, say 'I am sorry, but I do not have enough information to answer that.'"
    },
    {
        "role": "user",
        "content": """
Context documents:
{% for doc in documents %}
- [ID: {{ doc.id }}] {{ doc.text }}
{% endfor %}

Question: {{ user_query }}
"""
    }
]

rag_prompt_v2 = mlflow.genai.register_prompt(
    name="rag_research_assistant",
    template=updated_rag_chat_template,
    commit_message="Added strict fallback guardrails for missing info.",
    tags={"pipeline": "rag", "engine": "jinja2"}
    )
mlflow.genai.set_prompt_alias(name="rag_research_assistant", version=2, alias="production")

In [13]:
import os
from openai import OpenAI

client = OpenAI(api_key=os.environ.get("OPENAI_API_KEY"))

retrieved_docs = [
    {"id": "DOC-01", "text": "MLflow LLM Tracking allows logging parameters, metrics, and prompts."},
    {"id": "DOC-02", "text": "The MLflow Prompt Registry is an immutable store for managing prompt versions."}
]


prompt_template = mlflow.genai.load_prompt("prompts:/rag_research_assistant@production")

formatted_chat = prompt_template.format(
    document=retrieved_docs,
    user_query="Can I change a prompt version once it is registered?"
)

response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=formatted_chat,
    temperature=0.0
)

print(response.choices[0].message.content)

I am sorry, but I do not have enough information to answer that.
